# Task 4 — Direct Kafka-to-Neo4j ingestion

Evidence run metadata is read from `screenshots/stage2_manifest.json`. Command: `bash scripts/capture_store_evidence.sh`. Kafka Connect uses `org.neo4j.connectors.kafka.sink.Neo4jConnector`; Spark is deliberately absent from this graph route. Raw query output: [`node_count.txt`](../screenshots/neo4j/node_count.txt).


In [1]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'screenshots/stage2_manifest.json').exists())
m = json.loads((ROOT / 'screenshots/stage2_manifest.json').read_text())['metrics']['neo4j']
print('total nodes:', m['total_nodes'])
print('explicit CPG nodes:', m['explicit_nodes'])
print('CALL_UNRESOLVED placeholders:', m['placeholders'])
print('relationships:', m['edges'])
print('duplicate node/edge groups:', m['duplicate_node_groups'], '/', m['duplicate_edge_groups'])
assert m == {'total_nodes': 22628, 'explicit_nodes': 21415, 'placeholders': 1213, 'edges': 7968, 'duplicate_node_groups': 0, 'duplicate_edge_groups': 0}


total nodes: 22628
explicit CPG nodes: 21415
CALL_UNRESOLVED placeholders: 1213
relationships: 7968
duplicate node/edge groups: 0 / 0


## Final database UI evidence

The image below is real Neo4j Browser evidence from the canonical Stage 3 replay final state, after stale cleanup. Command/query: target `run_id` and graph count query. Run date: `2026-07-17`. Result: `pass`. The screenshot is labeled as replay final state rather than as the earlier Stage 2 baseline. Its values are backed by `screenshots/replay/neo4j_after.json` and the strict replay manifest.

![Neo4j Browser final state after Stage 3 replay cleanup](../screenshots/replay/neo4j_after_cleanup.png)


## Reflection

**Worked:** the Stage 2 connector reached `RUNNING`, upserted 21,415 explicit nodes and 7,968 relationships, and created 1,213 deterministic unresolved-call placeholders. **Failed:** that clean run had query artifacts but no dedicated Neo4j Browser capture. **Resolution:** the canonical Stage 3 replay captured a real, hash-validated Neo4j Browser final-state image after stale cleanup; it is presented above with its actual phase and does not replace the Stage 2 counts.
